In [ ]:
import pandas as pd

abcd = pd.read_parquet("data/processed/abcd_first_msgs.parquet")

display(abcd.head())
print("Table shape:", abcd.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/abcd_first_msgs.parquet'

In [ ]:
import pandas as pd

abcd = pd.read_parquet("data/processed/abcd_first_msgs.parquet")

In [ ]:
#predicting the refund column using the first_msgs column
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train = abcd[abcd["split"] == "train"]
dev = abcd[abcd["split"] == "dev"]

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(class_weight="balanced", max_iter=1000)
)

model.fit(train["first_msgs"].fillna(""), train["refund"])
print("Model trained")

Model trained


In [ ]:
predictions = model.predict(dev["first_msgs"].fillna(""))

results = dev[["id", "first_msgs", "refund"]].copy()
results["predicted_refund"] = predictions
results["correct"] = results["refund"] == results["predicted_refund"]

display(results.head(10))

,id,first_msgs,refund,predicted_refund,correct
8034,3532,I was trying to see my history but I can't rem...,0,0,True
8035,9036,Yes i would like to double check the status of...,0,0,True
8036,7177,"Yeah, this is Crystal Minh, my husband said th...",1,1,True
8037,3573,I would like to cancel my shipping order I ord...,0,1,False
8038,2062,Hi I need to check the status of my shipped or...,0,0,True
8039,7399,I need to pay my premium subscription fee.,0,0,True
8040,6899,hello i want to know about maintaining and kee...,0,0,True
8041,2163,"Hi, I would like to know about an item that is...",0,0,True
8042,8250,i need an extension on my subscription with th...,0,0,True
8043,5569,"Hi, I'm trying to confirm a shipping date, I h...",0,0,True


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"Validation accuracy: {accuracy_score(dev['refund'], predictions):.1%}")
print(classification_report(dev["refund"], predictions, zero_division=0))

print("Confusion matrix: rows = actual, columns = predicted; order = 0, 1")
print(confusion_matrix(dev["refund"], predictions, labels=[0, 1]))

majority = train["refund"].mode()[0]
print(f"Majority baseline: {(dev['refund'] == majority).mean():.1%}")

Validation accuracy: 90.8%
              precision    recall  f1-score   support

           0       0.98      0.91      0.95       917
           1       0.48      0.84      0.61        87

    accuracy                           0.91      1004
   macro avg       0.73      0.88      0.78      1004
weighted avg       0.94      0.91      0.92      1004

Confusion matrix: rows = actual, columns = predicted; order = 0, 1
[[839  78]
 [ 14  73]]
Majority baseline: 91.3%


In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"}

def clean_message(text):
    if pd.isna(text):
        return ""

    text = str(text).replace("'", "'").strip()

    text = text.replace("won't", "will not")
    text = text.replace("can't", "can not")
    text = text.replace("n't", " not")
    text = re.sub(r"\bho!", " ", text)

    text = re.sub(
        r"^(hi+|hello+|helo+|hey+|hiya|good morning|"
        r"good afternoon|good evening)\b[\s,.!]*",
        "", text
    )

    text = re.sub(r"[^\w\s]|_", " ", text)
    text = " ".join(word for word in text.split() if word not in stop_words)

    return text

abcd["clean_text"] = abcd["first_msgs"].apply(clean_message)

display(abcd[["first_msgs", "clean_text"]].head(10))

,first_msgs,clean_text
0,"Hi! I need to return an item, can you help me ...",Hi I need return item help
1,just wanted to check on the status of a refund,just wanted check status refund
2,HEY HO! I've got a promo code and I want to kn...,HEY HO I ve got promo code I want know expire ...
3,"Hello, I would like to change my shipping deat...",Hello I like change shipping deatails changed ...
4,Hello. I have a really cool party coming up. A...,Hello I really cool party coming And I need ne...
5,I want an extension for my shopping service. I...,I want extension shopping service I need time ...
6,"Hello, I am trying to check the status of my o...",Hello I trying check status order I forgot use...
7,Hello I am trying to buy a pair of boots but t...,Hello I trying buy pair boots not getting adde...
8,I accidently ordered the wrong item and was wo...,I accidently ordered wrong item wondering I ch...
9,I would like to confirm my address is correct ...,I like confirm address correct order Am I able


In [ ]:
abcd["tokens"] = abcd["clean_text"].str.split()

display(abcd[["clean_text", "tokens"]].head())

,clean_text,tokens
0,Hi I need return item help,"[Hi, I, need, return, item, help]"
1,just wanted check status refund,"[just, wanted, check, status, refund]"
2,HEY HO I ve got promo code I want know expire ...,"[HEY, HO, I, ve, got, promo, code, I, want, kn..."
3,Hello I like change shipping deatails changed ...,"[Hello, I, like, change, shipping, deatails, c..."
4,Hello I really cool party coming And I need ne...,"[Hello, I, really, cool, party, coming, And, I..."


In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/oscaralawi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
def tokenize_message(sentence):
    word_tokens = word_tokenize(sentence, preserve_line=True)
    word_tokens = [word for word in word_tokens if word not in stop_words]

    sentence = [
        lemmatizer.lemmatize(word, pos="n")
        for word in word_tokens
    ]
    return sentence

abcd["tokens"] = abcd["clean_text"].apply(tokenize_message)

display(abcd[["clean_text", "tokens"]].head())

,clean_text,tokens
0,Hi I need return item help,"[Hi, I, need, return, item, help]"
1,just wanted check status refund,"[just, wanted, check, status, refund]"
2,HEY HO I ve got promo code I want know expire ...,"[HEY, HO, I, ve, got, promo, code, I, want, kn..."
3,Hello I like change shipping deatails changed ...,"[Hello, I, like, change, shipping, deatails, c..."
4,Hello I really cool party coming And I need ne...,"[Hello, I, really, cool, party, coming, And, I..."


In [ ]:
train= abcd[abcd["split"] == "train"]

In [ ]:
abcd["token_text"] = abcd["tokens"].str.join(" ")

train = abcd[abcd["split"] == "train"].copy()
dev = abcd[abcd["split"] == "dev"].copy()

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)

X_train = vectorizer.fit_transform(train["token_text"])
X_dev = vectorizer.transform(dev["token_text"])

y_train = train["refund"]
y_dev = dev["refund"]

print("Vectorized training shape:", X_train.shape)

Vectorized training shape: (8034, 6682)


In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

classifier.fit(X_train, y_train)
predictions = classifier.predict(X_dev)

print("First 10 predictions:", predictions[:10])

First 10 predictions: [0 0 0 1 0 0 0 0 0 0]


In [ ]:
classifier.fit(X_train, y_train)
predictions = classifier.predict(X_dev)

In [ ]:
predictions = classifier.predict(X_dev)

In [ ]:
train = abcd.loc[abcd["split"] == "train"].copy()
dev = abcd.loc[abcd["split"] == "dev"].copy()

X_train = vectorizer.fit_transform(train["clean_text"])
X_dev = vectorizer.transform(dev["clean_text"])

classifier.fit(X_train, train["refund"])
predictions = classifier.predict(X_dev)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Show actual labels and predicted labels for the first 10 examples in the dev set
results = dev[["first_msgs", "clean_text", "refund"]].copy()
results["predicted_refund"] = predictions
results["correct"] = results["refund"] == results["predicted_refund"]

display(results.head(10))

# Measure performance on validation data
print(f"Validation accuracy: {accuracy_score(dev['refund'], predictions):.1%}")

print(classification_report(
    dev["refund"],
    predictions,
    zero_division=0
))

# Compare with always predicting the most common training label
majority_label = train["refund"].mode()[0]
print(f"Majority baseline: {(dev['refund'] == majority_label).mean():.1%}")

,first_msgs,clean_text,refund,predicted_refund,correct
8034,I was trying to see my history but I can't rem...,I trying history I not remember username,0,0,True
8035,Yes i would like to double check the status of...,Yes like double check status refund,0,0,True
8036,"Yeah, this is Crystal Minh, my husband said th...",Yeah Crystal Minh husband said subscription ad...,1,0,False
8037,I would like to cancel my shipping order I ord...,I like cancel shipping order I ordered wrong item,0,1,False
8038,Hi I need to check the status of my shipped or...,Hi I need check status shipped order,0,0,True
8039,I need to pay my premium subscription fee.,I need pay premium subscription fee,0,0,True
8040,hello i want to know about maintaining and kee...,want know maintaining keeping subscription wan...,0,0,True
8041,"Hi, I would like to know about an item that is...",Hi I like know item stock,0,0,True
8042,i need an extension on my subscription with th...,need extension subscription corona jsut dont m...,0,0,True
8043,"Hi, I'm trying to confirm a shipping date, I h...",Hi I m trying confirm shipping date I contradi...,0,0,True


Validation accuracy: 91.0%
              precision    recall  f1-score   support

           0       0.99      0.91      0.95       917
           1       0.49      0.86      0.62        87

    accuracy                           0.91      1004
   macro avg       0.74      0.89      0.79      1004
weighted avg       0.94      0.91      0.92      1004

Majority baseline: 91.3%
